# Redrob Hackathon — Candidate Ranking Sandbox

**Team:** Jashwanth S (Solo)  
**Problem:** Intelligent Candidate Discovery & Ranking Challenge  
**Goal:** Rank candidates for Senior AI Engineer using multi-signal scoring + XGBoost LTR

---

### Sandbox Mode (Default)
Runs the full pipeline on 50 sample candidates — CPU only, no GPU needed.  
Demonstrates: feature extraction → embedding → BM25 → cross-encoder → XGBoost LTR → reasoning generation.

### Full Pipeline Mode (Optional — Section 2)
Runs on all 100K candidates — requires T4 GPU + dataset extraction.  
Produces the actual submission CSV.

---
## Section 1 — Sandbox Demo (CPU, ~3 min)

In [ ]:
# Step 1: Clone repo
!git clone https://github.com/JASHWANTHS07/india-runs.git 2>/dev/null || echo 'Repo already cloned'
%cd india-runs

In [ ]:
# Step 2: Install dependencies
!pip install -q sentence-transformers tqdm pandas pyarrow xgboost optuna
print('Dependencies installed.')

In [ ]:
# Step 3: Run full pipeline on sample candidates (CPU, no GPU needed)
!python run_pipeline.py \
    --run_sample dataset/India_runs_data_and_ai_challenge/sample_candidates.json \
    --artifacts  ./artifacts_sample \
    --out        ./sandbox_output.csv \
    --cpu \
    --method     xgboost \
    --top-k      10

In [ ]:
# Step 4: Display results
import pandas as pd
pd.set_option('display.max_colwidth', 120)

df = pd.read_csv('sandbox_output.csv')
print(f'Candidates ranked: {len(df)}')
print(f'Score range: {df["score"].max():.4f} -> {df["score"].min():.4f}')
print()
df[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# Step 5: Validate output format
assert list(df.columns) == ['candidate_id', 'rank', 'score', 'reasoning'], f'Bad columns: {list(df.columns)}'
assert set(df['rank']) == set(range(1, len(df) + 1)), 'Ranks not contiguous'
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), 'Scores not non-increasing'
assert df['candidate_id'].is_unique, 'Duplicate candidate_ids'
assert df['reasoning'].notna().all(), 'Missing reasoning'

# Check reasoning diversity (no two identical)
unique_reasons = df['reasoning'].nunique()
print(f'Unique reasoning strings: {unique_reasons}/{len(df)}')
assert unique_reasons == len(df), 'Duplicate reasoning detected'

print('All validation checks passed.')

In [ ]:
# Step 6: Run test suite
!python -m pytest tests/ -v --tb=short 2>&1 | tail -20

---
## Section 2 — Full 100K Pipeline (GPU Required)

**Prerequisites:** T4 GPU runtime + dataset extracted from Data.rar

Skip this section for sandbox review — it produces the actual submission CSV.

In [ ]:
# GPU check
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU! Go to Runtime -> Change runtime type -> T4 GPU')
print(result.stdout.split('\n')[0])
print('GPU OK')

In [ ]:
# Extract dataset (run once)
!apt-get install -qq unrar
!unrar x -o+ dataset/Data.rar dataset/
print('\nExtraction complete.')
!ls dataset/India_runs_data_and_ai_challenge/

In [ ]:
# Verify dataset
import os
CANDIDATES = 'dataset/India_runs_data_and_ai_challenge/candidates.jsonl'
size_mb = os.path.getsize(CANDIDATES) / 1e6
print(f'candidates.jsonl: {size_mb:.1f} MB')
assert size_mb > 400, f'Too small ({size_mb:.1f} MB)'
print('Dataset OK')

In [ ]:
# Pre-compute (GPU) — ~15 min on T4
# Embeds 100K candidates with BAAI/bge-large-en-v1.5, computes BM25,
# cross-encoder re-ranks top-1000, extracts 76 features per candidate.
!python src/precompute.py \
    --candidates dataset/India_runs_data_and_ai_challenge/candidates.jsonl \
    --artifacts  ./artifacts

In [ ]:
# Verify artifacts
import numpy as np
import pandas as pd

emb = np.load('artifacts/embeddings.npy')
jd  = np.load('artifacts/jd_embedding.npy')
feat_df = pd.read_parquet('artifacts/features.parquet')

print(f'embeddings.npy       : {emb.shape}  ({emb.nbytes/1e6:.0f} MB)')
print(f'jd_embedding         : {jd.shape}')
print(f'features.parquet     : {len(feat_df)} rows, {len(feat_df.columns)} cols')
print(f'  bm25 range         : [{feat_df.bm25_score.min():.3f}, {feat_df.bm25_score.max():.3f}]')
print(f'  cross-encoder range: [{feat_df.cross_encoder_score.min():.3f}, {feat_df.cross_encoder_score.max():.3f}]')
print('Artifacts OK')

In [ ]:
# Rank (CPU, XGBoost LTR) — <3 min
# Multiplicative scoring: relevance_gate x technical_core x fit_multiplier x behavioral x availability
# XGBoost rank:pairwise on 78 features, blended 60/40 with heuristic scores.
!python src/rank.py \
    --artifacts ./artifacts \
    --out       ./jashwanth_s.csv \
    --method    xgboost \
    --tune      --tune-trials 50

In [ ]:
# Validate submission
import pandas as pd
from dataclasses import fields
import sys
sys.path.insert(0, '.')

df = pd.read_csv('jashwanth_s.csv')
assert len(df) == 100, f'Expected 100 rows, got {len(df)}'
assert set(df['rank']) == set(range(1, 101)), 'Ranks not 1-100'
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), 'Scores not non-increasing'

# Honeypot check
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
feat_df = pd.read_parquet('artifacts/features.parquet')
top_feats = feat_df[feat_df['candidate_id'].isin(set(df['candidate_id']))]
honeypots = []
for _, row in top_feats.iterrows():
    kwargs = {f.name: row[f.name] for f in fields(CandidateFeatures) if f.name in row.index}
    kwargs.setdefault('profile_text', '')
    if is_honeypot(CandidateFeatures(**kwargs)):
        honeypots.append(row['candidate_id'])

print(f'Rows      : {len(df)}')
print(f'Scores    : {scores[0]:.4f} -> {scores[-1]:.4f}')
print(f'Honeypots : {len(honeypots)} (must be 0)')
assert len(honeypots) == 0, f'Honeypots in top 100: {honeypots}'
print('All checks passed.')

In [ ]:
# Display top 10
pd.set_option('display.max_colwidth', 120)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# Download submission CSV + artifacts
from google.colab import files
files.download('jashwanth_s.csv')
print('Downloaded.')